# Topic Classification of Real User Questions — Yahoo! Answers (10 Classes) — COMPLETE RUN

**CSE440 NLP II Lab Project — finished, end-to-end runnable version** (built for: Intel Arc B580 12GB VRAM · 16GB RAM · Windows)

| | |
|---|---|
| Group | [Group No.] |
| Members | [Name (ID)] · [Name (ID)] · [Name (ID)] · [Name (ID)] |

**Task.** 10-way classification of noisy user-submitted questions (title + content) into: Society & Culture, Science & Mathematics, Health, Education & Reference, Computers & Internet, Sports, Business & Finance, Entertainment, Family & Relationships, Politics & Government.

**Spec compliance.** 10 classes (≥ 4 required; within the 5–10 project band) · 27,500 samples used (≥ 10,000 required), seeded stratified subsample of the official 1.46M · English · train/val/test splits · balanced by design · non-trivial noisy-UGC topic classification.

**Hardware notes (why this notebook is built this way).**
- Neural models use **PyTorch**, not TensorFlow/Keras — stock TensorFlow cannot use an Intel Arc GPU, while PyTorch has XPU builds that can (`pip install torch --index-url https://download.pytorch.org/whl/xpu`). Device auto-detection: CUDA → XPU → CPU.
- **Cloud runtimes (Kaggle/Colab):** runs unchanged on a CUDA GPU accelerator — the auto-detect picks `cuda` and torch is preinstalled (no install cell needed). On Kaggle, either attach the original CSVs as a private dataset (auto-discovered under `/kaggle/input/...`) or attach nothing and the notebook downloads the HF mirror; tables and figures are written under `/kaggle/working` so they are saved with the notebook version.
- 16GB RAM: the 1.4M-row corpus is drawn per-class from the memory-mapped Arrow dataset and only the ~27.5k sampled rows are ever materialized as pandas.
- BERT runs at max length 96 and batch 16–32 (~4–6GB VRAM) — comfortable on 12GB.
- Expected wall time on the B580: classical + RNN sections ≈ 30–60 min, BERT section ≈ 1.5–2.5 h. On CPU fallback everything still runs, but BERT becomes impractical (≳ a day).
- Subsample sizes are constants in the dataset cell — raise them if you re-run on stronger hardware, but keep the total ≥ 10,000.

## 0. Environment Setup

In [ ]:
# ====================================================================
# OPTIONAL: PyTorch GPU/Accelerator Installation (Run once if needed)
# ====================================================================

# --- For cloud GPU runtimes (Kaggle / Colab) ---
# torch with CUDA support is already preinstalled - nothing to install.

# --- For NVIDIA GPUs (CUDA 12.4) ---
# %pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# --- For Intel Arc GPUs (XPU e.g. B580, A770, A750) ---
# %pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/xpu

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = ['datasets', 'gensim', 'wordcloud', 'transformers', 'nltk', 'tqdm']
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Package check complete')

# PyTorch GPU support (install once, in a terminal, matching your GPU):
#   NVIDIA CUDA - the default wheel already includes CUDA: pip install torch
#   Intel Arc XPU - pip install torch --index-url https://download.pytorch.org/whl/xpu


In [ ]:
import os
import re
import glob
import random
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Cloud runtimes: on Kaggle only /kaggle/working is persisted as notebook output,
# so results are redirected there; locally the repo-root data/ directory is used.
ON_KAGGLE = os.path.exists('/kaggle/input') or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ROOT_DIR = '/kaggle/working' if ON_KAGGLE else '..'
OUT_DIR = os.path.join(ROOT_DIR, 'data', 'processed')
FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 100)

print('pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

FORCE_DEVICE = None
def get_device(force: str | None = None) -> torch.device:
    if force:
        return torch.device(force)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        return torch.device("xpu")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")
def set_seed(seed: int = 42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        torch.xpu.manual_seed(seed)
DEVICE = get_device(FORCE_DEVICE)
set_seed(SEED)
device_name = ""
if DEVICE.type == "cuda":
    device_name = f" ({torch.cuda.get_device_name(0)})"
elif DEVICE.type == "xpu":
    device_name = f" ({torch.xpu.get_device_name(0)})"
print(f"PyTorch {torch.__version__} | Using device: {DEVICE}{device_name}")
if DEVICE.type == "cpu":
    print("\n⚠️ WARNING: No GPU/XPU accelerator detected. Training/Inference will be slow.")
    print("👉 For Intel Arc GPUs (e.g., B580), ensure the Intel XPU PyTorch wheel is installed.")

## 1. Problem Definition *(spec §3.1)*

- **Input:** a user-submitted question (title + body) from a community Q&A platform — raw, unedited, noisy.
- **Output:** one of 10 topical categories.
- **Real-world relevance:** community platforms must route new questions to the right category or expert pool; the same classifier feeds search, moderation, FAQ matching, and knowledge-base organization. The practically hard part is doing this on *unfiltered user text* — misspellings, slang, URLs.
- **Research questions:** (1) how much do preprocessing depth and noise cleanup matter on user-generated text? (2) how far do sparse lexical features (TF-IDF) go versus dense embeddings (Word2Vec trained here vs pretrained GloVe) when vocabulary is noisy? (3) does recurrence direction (unidirectional vs bidirectional) matter for topic evidence scattered across a question? (4) how large is the gap between pretrained contextual (BERT) and non-contextual representations on noisy UGC?

## 2. Dataset Collection and Description *(spec §3.2)*

- **Local files:** `Dataset/train.csv` (746MB) and `Dataset/test.csv` (32MB) — the original Zhang et al. Yahoo! Answers corpus, headerless CSV with positional columns: 0 = class index, 1 = question title, 2 = question content, 3 = best answer. Verified: 1,400,000 train / 60,000 test rows; classes 1–10 with ~140k train rows each.
- **Original:** Zhang, X., Zhao, J., and LeCun, Y. 2015. *Character-level Convolutional Networks for Text Classification.* NeurIPS 2015. (Convenient mirror: [Yahoo_Answers_10_categories_for_NLP on HF](https://huggingface.co/datasets/yassiracharki/Yahoo_Answers_10_categories_for_NLP).)
- **Subsampling plan (documented, reproducible):** training 10 models incl. BERT on 1.46M rows is infeasible on this hardware, so we draw a **seeded stratified subsample** — 2,000 train + 250 validation per class from the official train pool, 500 test per class from the **official test set** — 20,000 / 2,500 / 5,000 = 27,500 samples (≥ 10,000 required). Class balance is preserved by construction; draws use `np.random.RandomState(SEED)`.
- **Memory management:** only the three needed columns are read (`best_answer` skipped — it dominates file size and is unused), and the 1.4M-row frames are deleted immediately after the draw, keeping peak RAM around 2GB.
- **Fields used:** `question_title` + `question_content` (`best_answer` noted as future work in §12).

In [ ]:
DATASET_ID = 'yassiracharki/Yahoo_Answers_10_categories_for_NLP'
CSV_NAMES = ['class_index', 'question_title', 'question_content']
LABEL_COL = 'class_index'
TEXT_COLS = ['question_title', 'question_content']

N_PER_CLASS_TRAIN = 4000
N_PER_CLASS_VAL = 500
N_PER_CLASS_TEST = 1000

# # if training takes way too long use these
# N_PER_CLASS_TRAIN = 2000
# N_PER_CLASS_VAL = 250
# N_PER_CLASS_TEST = 500

# The notebook prefers the local original CSVs (headerless; positional columns
# 0 class_index, 1 question_title, 2 question_content, 3 best_answer) and
# searches common locations; on any other machine (e.g. Colab/Kaggle) it falls
# back to downloading the Hugging Face mirror via the datasets library.
def find_csv(name):
    candidates = [
        os.path.join('..', 'Dataset', name),
        os.path.join('Dataset', name),
        os.path.join('data', 'raw', name),
        os.path.join('/content', 'Dataset', name),
    ]
    # Kaggle: attached datasets are mounted under /kaggle/input/<dataset-slug>/...
    candidates += glob.glob('/kaggle/input/**/' + name, recursive=True)
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return None

def read_csv_columns(path):
    return pd.read_csv(path, header=None, names=CSV_NAMES, usecols=[0, 1, 2],
                       dtype={'class_index': np.int16}, encoding='utf-8', encoding_errors='replace')

train_csv, test_csv = find_csv('train.csv'), find_csv('test.csv')
if train_csv and test_csv:
    print('Loading local CSVs:', train_csv, '|', test_csv)
    train_full = read_csv_columns(train_csv)
    test_full = read_csv_columns(test_csv)
else:
    print('Local CSVs not found - downloading the HF mirror:', DATASET_ID)
    from datasets import load_dataset
    raw = load_dataset(DATASET_ID)
    train_full = raw['train'].select_columns([LABEL_COL] + TEXT_COLS).to_pandas()
    test_full = raw['test'].select_columns([LABEL_COL] + TEXT_COLS).to_pandas()
    del raw

LABEL_CODES = sorted(train_full[LABEL_COL].unique().tolist())
print('label codes:', LABEL_CODES, '| train rows:', len(train_full), '| test rows:', len(test_full))

def stratified_draw(frame, per_class, seed):
    rng = np.random.RandomState(seed)
    picked = []
    for code in LABEL_CODES:
        candidates = np.where(frame[LABEL_COL].values == code)[0]
        rng.shuffle(candidates)
        picked.extend(candidates[:per_class].tolist())
    return frame.iloc[picked].reset_index(drop=True)

train_val_df = stratified_draw(train_full, N_PER_CLASS_TRAIN + N_PER_CLASS_VAL, SEED)
shuffled = train_val_df.sample(frac=1.0, random_state=SEED)
val_part = shuffled.groupby(LABEL_COL, group_keys=False).head(N_PER_CLASS_VAL)
train_part = shuffled.drop(val_part.index)
test_part = stratified_draw(test_full, N_PER_CLASS_TEST, SEED)
del train_full, test_full, train_val_df, shuffled

print('train:', train_part.shape, '| val:', val_part.shape, '| test:', test_part.shape)

In [ ]:
TOPIC_NAMES_BY_CODE = {
    1: 'Society & Culture',
    2: 'Science & Mathematics',
    3: 'Health',
    4: 'Education & Reference',
    5: 'Computers & Internet',
    6: 'Sports',
    7: 'Business & Finance',
    8: 'Entertainment',
    9: 'Family & Relationships',
    10: 'Politics & Government',
}
if min(LABEL_CODES) == 0:
    TOPIC_NAMES_BY_CODE = {code - 1: name for code, name in TOPIC_NAMES_BY_CODE.items()}

def build_text(frame):
    merged = frame[TEXT_COLS[0]].fillna('').astype(str)
    for col in TEXT_COLS[1:]:
        merged = merged + ' ' + frame[col].fillna('').astype(str)
    return merged.str.replace(r'\s+', ' ', regex=True).str.strip()

for frame in (train_part, val_part, test_part):
    frame['text'] = build_text(frame)

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder().fit(train_part[LABEL_COL])
CLASS_NAMES = [TOPIC_NAMES_BY_CODE[code] for code in label_encoder.classes_]
y_train = label_encoder.transform(train_part[LABEL_COL])
y_val = label_encoder.transform(val_part[LABEL_COL])
y_test = label_encoder.transform(test_part[LABEL_COL])

print('Classes (encoded order):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i}: {name}')
train_part[['text']].head()

## 3. Exploratory Data Analysis *(spec §3.3)*

Classes are balanced by construction — so the informative EDA is text-side: length distributions (they justify `MAX_LEN` later), noise (URLs, HTML remnants), duplicates, and per-topic vocabulary.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (name, frame) in zip(axes, [('train', train_part), ('validation', val_part), ('test', test_part)]):
    counts = frame[LABEL_COL].map(TOPIC_NAMES_BY_CODE).value_counts()
    sns.barplot(x=counts.values, y=counts.index, ax=ax)
    ax.set_title(f'{name} (n={len(frame)})')
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'class_distribution.png'), dpi=200, bbox_inches='tight')
plt.show()
pd.concat(
    [frame[LABEL_COL].map(TOPIC_NAMES_BY_CODE).value_counts(normalize=True).rename(name)
     for name, frame in [('train', train_part), ('val', val_part), ('test', test_part)]],
    axis=1,
)

In [ ]:
train_part['n_words'] = train_part['text'].str.split().str.len()
train_part['n_chars'] = train_part['text'].str.len()

display(train_part['n_words'].describe().to_frame().T)
print('word-count percentiles:')
print(train_part['n_words'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_part['n_words'].clip(upper=400), bins=50, ax=axes[0])
axes[0].set_title('Question length (words, clipped at 400)')
sns.boxplot(data=train_part, x=train_part[LABEL_COL].map(TOPIC_NAMES_BY_CODE), y='n_words', ax=axes[1])
axes[1].tick_params(axis='x', rotation=60)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'length_stats.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
from wordcloud import WordCloud

ordered_topics = [TOPIC_NAMES_BY_CODE[code] for code in sorted(TOPIC_NAMES_BY_CODE)]
fig, axes = plt.subplots(2, 5, figsize=(19, 7))
for ax, topic in zip(axes.flat, ordered_topics):
    text = ' '.join(train_part.loc[train_part[LABEL_COL].map(TOPIC_NAMES_BY_CODE) == topic, 'text'])
    cloud = WordCloud(width=520, height=360, background_color='white').generate(text)
    ax.imshow(cloud)
    ax.set_title(topic)
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'wordclouds.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
print('Missing values per field:')
print(train_part[TEXT_COLS].isna().sum().to_string())
print('Empty combined texts:', (train_part['text'].str.len() == 0).sum())
print('Duplicate combined texts:', train_part['text'].duplicated().sum())
print('Rows containing URLs:', train_part['text'].str.contains(r'https?://|www\.', regex=True).sum())
print('Rows containing HTML entities:', train_part['text'].str.contains(r'&\w+;', regex=True).sum())

noisy = train_part.loc[train_part['text'].str.contains(r'https?://|&\w+;', regex=True), ['text']]
if len(noisy):
    display(noisy.head(3))

## 4. Data Preprocessing *(spec §3.4)*

Three strategies of increasing depth, compared empirically on a validation-trained baseline before fixing the final choice:

| Strategy | Steps | Rationale |
|---|---|---|
| minimal | lowercase, URL removal, HTML-entity removal, whitespace normalization | removes platform artifacts, preserves all wording |
| standard | minimal + punctuation stripping + stopword removal | tests whether generic stopwords help short questions |
| lemmatized | standard + lemmatization | tests whether morphological normalization helps noisy UGC |

Tension worth reporting: aggressive cleanup can destroy topic evidence — e.g. URLs can *be* the signal for Computers & Internet.

In [ ]:
import nltk

for resource in ['stopwords', 'wordnet', 'omw-1.4']:
    try:
        nltk.data.find('corpora/' + resource)
    except LookupError:
        nltk.download(resource, quiet=True)

from nltk.corpus import stopwords as nltk_stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS_EN = set(nltk_stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def preprocess_minimal(text):
    t = text.lower()
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    t = re.sub(r'&\w+;', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def preprocess_standard(text):
    t = preprocess_minimal(text).translate(str.maketrans('', '', string.punctuation))
    tokens = [w for w in t.split() if w and w not in STOPWORDS_EN]
    return ' '.join(tokens)

def preprocess_lemmatized(text):
    tokens = [LEMMATIZER.lemmatize(w) for w in preprocess_standard(text).split()]
    return ' '.join(tokens)

example = train_part['text'].iloc[0]
print('ORIGINAL  :', example[:180])
print('MINIMAL   :', preprocess_minimal(example)[:180])
print('STANDARD  :', preprocess_standard(example)[:180])
print('LEMMATIZED:', preprocess_lemmatized(example)[:180])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

STRATEGIES = {
    'minimal': preprocess_minimal,
    'standard': preprocess_standard,
    'lemmatized': preprocess_lemmatized,
}

strategy_scores = []
for name, fn in STRATEGIES.items():
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000, sublinear_tf=True)
    Xtr = vectorizer.fit_transform(train_part['text'].apply(fn))
    Xva = vectorizer.transform(val_part['text'].apply(fn))
    model = MultinomialNB(alpha=0.5).fit(Xtr, y_train)
    pred = model.predict(Xva)
    strategy_scores.append({
        'strategy': name,
        'val_accuracy': accuracy_score(y_val, pred),
        'val_macro_f1': f1_score(y_val, pred, average='macro'),
    })

strategy_table = pd.DataFrame(strategy_scores).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
display(strategy_table)

SELECTED_STRATEGY = strategy_table.iloc[0]['strategy']
print('Selected preprocessing strategy (highest validation macro-F1):', SELECTED_STRATEGY)

In [ ]:
clean_fn = STRATEGIES[SELECTED_STRATEGY]
X_train_text = train_part['text'].apply(clean_fn).values
X_val_text = val_part['text'].apply(clean_fn).values
X_test_text = test_part['text'].apply(clean_fn).values

pd.DataFrame({'selected-strategy text': X_train_text[:5]})

## 5. Train / Validation / Test Split *(spec §3.5)*

Splits come from §2: the **official test set** (subsampled, untouched during training and tuning) and a **stratified validation set** carved from the official train pool. All draws are seeded → fully reproducible; class proportions are identical across splits.

In [ ]:
for name, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    counts = np.bincount(y, minlength=len(CLASS_NAMES))
    print(f'{name:5s} n={len(y):6d} | per-class: {counts.tolist()} | min share: {counts.min() / len(y):.3f}')

## 6. Text Representations 

1. **TF-IDF** (sparse, unigrams+bigrams) — feeds Logistic Regression and Multinomial NB directly; for Random Forest we reduce it to 200 dense LSA dimensions (`TruncatedSVD`), because tree ensembles degrade on very wide sparse spaces.
2. **Word2Vec** — trained from scratch on the training corpus (skip-gram); questions → mean word vectors.
3. **GloVe** — pretrained `glove-wiki-gigaword-100`; questions → mean word vectors + OOV coverage report.
4. **Char n-grams (3–5)** — `char_wb` TF-IDF variant on minimally cleaned text; robustness check for the noisy spelling found in the EDA (word-level features die on misspellings; subword features survive).

### 6.1 Design Rationale — Why Each Representation Exists

The pipeline builds **one sparse-lexical family with three variants** plus **two true word embeddings**. SVD and char n-grams are not embeddings — they are deliberate variants of TF-IDF, each serving a specific model or research question.

| # | Representation | What it is | Consumed by | Why it exists |
|---|---|---|---|---|
| 1 | **TF-IDF (words)** | sparse unigram+bigram importance weights | LogReg, NB | linear models learn one weight per feature — wide sparse input is their native habitat |
| 2 | **LSA = TF-IDF → SVD(200)** | the same TF-IDF compressed to 200 dense latent dimensions | Random Forest only | trees split on one feature at a time and cannot digest ~35k mostly-zero columns; 200 dense dimensions are the format trees need |
| 3 | **TF-IDF (char 3–5)** | letter-fragment variant on minimally cleaned text | LogReg, NB | word features die on misspellings and rare words (EDA: *"haw to open avi files?"*); subword fragments degrade gracefully instead |
| 4 | **Word2Vec / GloVe** | mean-pooled dense word vectors | LogReg, NB, RF | answers research question 2 — how far sparse lexical features go versus dense embeddings on noisy UGC |

**Notes from this run.**
- LSA retains only ~15% of TF-IDF variance at 200 dimensions. That compression is why Random Forest (~0.49) trails Logistic Regression on full TF-IDF (~0.64) — the gap is a *finding*: aggressive dimensionality reduction discards too much topic signal for any model.
- LogReg and NB never use SVD — they always receive the uncompressed TF-IDF (or its char variant).
- A typo like *haw* kills only that one word-level feature, while the rest of the question still contributes matching fragments; morphological variants (*file / files / filing*) collapse onto shared fragments.

**One-line summary:** SVD exists because tree models cannot handle wide sparse TF-IDF; char n-grams exist because word features are brittle under user misspellings — both are design choices tied to specific models and research questions, not additional embeddings.


In [ ]:
# Character n-gram TF-IDF - robustness check for noisy spelling
# (EDA: 'haw to open avi files?'). char_wb keeps n-grams inside words;
# built on minimally cleaned text because char n-grams exploit word
# boundaries and punctuation, which the lemmatized strategy strips.
char_text_train = train_part['text'].apply(preprocess_minimal)
char_text_val = val_part['text'].apply(preprocess_minimal)
char_text_test = test_part['text'].apply(preprocess_minimal)

char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                             min_df=3, max_features=200000, sublinear_tf=True)
X_train_char = char_tfidf.fit_transform(char_text_train)
X_val_char = char_tfidf.transform(char_text_val)
X_test_char = char_tfidf.transform(char_text_test)
print('char TF-IDF:', X_train_char.shape)


In [ ]:
from sklearn.decomposition import TruncatedSVD

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)
X_test_tfidf = tfidf.transform(X_test_text)

svd = TruncatedSVD(n_components=200, random_state=SEED)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_val_svd = svd.transform(X_val_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print('TF-IDF:', X_train_tfidf.shape, '| LSA:', X_train_svd.shape,
      f'| LSA variance retained: {svd.explained_variance_ratio_.sum():.3f}')

In [ ]:
import gensim
from gensim.models import Word2Vec

tokenized_train = [t.split() for t in X_train_text]
tokenized_val = [t.split() for t in X_val_text]
tokenized_test = [t.split() for t in X_test_text]

w2v_model = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=2, sg=1, epochs=10, seed=SEED)

def mean_sentence_vector(tokens, keyed_vectors):
    vectors = [keyed_vectors[t] for t in tokens if t in keyed_vectors]
    return np.mean(vectors, axis=0) if vectors else np.zeros(keyed_vectors.vector_size)

X_train_w2v = np.vstack([mean_sentence_vector(t, w2v_model.wv) for t in tokenized_train])
X_val_w2v = np.vstack([mean_sentence_vector(t, w2v_model.wv) for t in tokenized_val])
X_test_w2v = np.vstack([mean_sentence_vector(t, w2v_model.wv) for t in tokenized_test])

print('Word2Vec vocabulary:', len(w2v_model.wv), '| feature matrices:', X_train_w2v.shape)
for probe in ['computer', 'school', 'game']:
    if probe in w2v_model.wv:
        print('nearest to', probe, ':', [w for w, _ in w2v_model.wv.most_similar(probe)[:5]])

In [ ]:
import gensim.downloader

glove = gensim.downloader.load('glove-wiki-gigaword-100')
X_train_glove = np.vstack([mean_sentence_vector(t, glove) for t in tokenized_train])
X_val_glove = np.vstack([mean_sentence_vector(t, glove) for t in tokenized_val])
X_test_glove = np.vstack([mean_sentence_vector(t, glove) for t in tokenized_test])

coverage = np.mean([any(t in glove for t in tokens) for tokens in tokenized_train])
print('GloVe matrix:', glove.vectors.shape, '| train questions with >=1 known token:', round(float(coverage), 3))

## 7. Classical Models *(spec §3.7 — Random Forest, Logistic Regression, Naive Bayes)*

Every run goes into `tuning_runs` (the dedicated §3.8 table). Each model gets **≥ 3 configurations** on its primary representation, plus runs on dense embeddings to compare representations.

### Change log — what we changed, why, and what it did

**Classical models (this section).** Validation numbers from the current 40k/5k run unless noted.

| Change | Before | After | Why | What it did |
|---|---|---|---|---|
| LogReg solver | liblinear (one-vs-rest) | lbfgs (multinomial) | lbfgs optimizes the true multinomial loss directly; liblinear can only do OvR for 10 classes | word-TF-IDF LogReg best at C=1.0 → 0.644 acc / 0.642 macro-F1 |
| LogReg C grid | {0.1, 1, 10} | {0.1, 0.5, 1, 2, 5} | coarse grid peaked at the interior (C=1) → refine around the optimum | plateau at C≈1–2 (0.644–0.645); both extremes lose 2–4 pts |
| NB alpha grid | {0.1, 0.5, 1.0} | {0.1, 0.3, 0.5, 0.7, 1.0} | alpha=0.1 won the coarse grid → probe the low-smoothing region | flat 0.651–0.653 across α=0.3–1.0 → MultinomialNB robust to smoothing; best 0.6534 (α=0.3) |
| **Char n-gram representation** | — | char_wb(3–5) TF-IDF on minimal-cleaned text | EDA noise ('haw to open avi files?'): word-level features fragment on misspellings; subword n-grams survive them | NB+char 0.638. LogReg+char was the **best classical model** in the earlier full execution (≈0.657 acc / 0.655 macro-F1, ~+1 pt over word TF-IDF) — the LogReg cell above now includes this run; execute it to reproduce the row |
| Word2Vec min_count | 5 | 2 | keep rare/noisy tokens in the from-scratch embeddings | vocabulary 21.8k on the 40k subsample; LogReg+w2v 0.621 |
| Random Forest | unchanged | unchanged | SVD(200) retains only 15% of TF-IDF variance — too lossy for trees | SVD-fed RF stuck ≈0.49–0.50; w2v-fed RF 0.612 acc / 0.607 macro-F1 is its best |

Takeaway: the classical ceiling sits at ≈0.65–0.66, and the **character representation is the one change that moved it** — direct evidence for the noisy-spelling finding from the EDA.

**RNN readout fix (§8, applied mid-project).**

- **Symptom:** SimpleRNN pinned at the 10% class prior (`val_f1 ≈ 0.02`, early stop in every configuration) while GRU/LSTM trained normally.
- **Cause:** `out[:, -1, :]` reads timestep 149, which is ~110 steps *after* the words. With post-padding feeding zero vectors, the un-gated recurrence contracts toward an input-independent fixed point, so all classes produce nearly the same classifier input.
- **Fix:** masked mean pooling over non-padding positions — every real token's state contributes, and each BPTT path from the loss to a word shortens dramatically. Applied to all 6 variants (the purge cell before the training loop keeps `tuning_runs` free of duplicate rows on re-runs; the LogReg cell above carries the same guard).
- **Verified offline (CPU, synthetic):** task with class evidence at t=0 under 138 trailing pads — new readout trains to 100% vs 36% for the old readout; unidirectional logits are exactly pad-invariant; gradients flow for all 6 cell/direction combos.
- **Expected:** SimpleRNN recovers well above chance but still trails GRU/LSTM → a clean "gating matters" ablation. Residual limitation: the backward direction of Bi- variants still consumes the padding first (`pack_padded_sequence` would remove that; see §12).

*Also added:* tqdm progress bars on RNN/BERT training and eval loops (visibility only, no math change).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.ensemble import RandomForestClassifier

REPS = {
    'tfidf': (X_train_tfidf, X_val_tfidf, X_test_tfidf),
    'svd': (X_train_svd, X_val_svd, X_test_svd),
    'w2v': (X_train_w2v, X_val_w2v, X_test_w2v),
    'glove': (X_train_glove, X_val_glove, X_test_glove),
    'char': (X_train_char, X_val_char, X_test_char),
}

tuning_runs = []
classical_specs = []

def log_classical_run(model_name, rep, config, factory):
    X_tr, X_va, _ = REPS[rep]
    model = factory().fit(X_tr, y_train)
    pred = model.predict(X_va)
    tuning_runs.append({
        'model': model_name,
        'representation': rep,
        'config': config,
        'val_accuracy': accuracy_score(y_val, pred),
        'val_macro_f1': f1_score(y_val, pred, average='macro'),
    })
    classical_specs.append({'model': model_name, 'representation': rep, 'config': config, 'factory': factory})

print('helpers ready')

In [ ]:
# Idempotent re-run: drop this model's earlier rows before appending fresh ones
# (no-op on a clean full run - tuning_runs is empty at this point).
tuning_runs[:] = [r for r in tuning_runs if r['model'] != 'Logistic Regression']
classical_specs[:] = [s for s in classical_specs if s['model'] != 'Logistic Regression']

for C in [0.1, 0.5, 1.0, 2.0, 5.0]:
    log_classical_run(
        'Logistic Regression', 'tfidf', 'C=' + str(C) + '',
        lambda C=C: LogisticRegression(C=C, solver='lbfgs', max_iter=1000, random_state=SEED),
    )
for rep in ['w2v', 'glove']:
    log_classical_run(
        'Logistic Regression', rep, 'C=1.0',
        lambda: LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=SEED),
    )
log_classical_run(
    'Logistic Regression', 'char', 'C=1.0, char_wb(3-5)',
    lambda: LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=SEED),
)
pd.DataFrame(tuning_runs)

In [ ]:
for alpha in [0.1, 0.3, 0.5, 0.7, 1.0]:
    log_classical_run(
        'Naive Bayes', 'tfidf', 'MultinomialNB alpha=' + str(alpha),
        lambda alpha=alpha: MultinomialNB(alpha=alpha),
    )
for smoothing in [1e-9, 1e-8, 1e-7]:
    log_classical_run(
        'Naive Bayes', 'w2v', 'GaussianNB var_smoothing=' + str(smoothing),
        lambda smoothing=smoothing: GaussianNB(var_smoothing=smoothing),
    )
log_classical_run(
    'Naive Bayes', 'glove', 'GaussianNB var_smoothing=1e-8',
    lambda: GaussianNB(var_smoothing=1e-8),
)
log_classical_run(
    'Naive Bayes', 'char', 'MultinomialNB alpha=0.5, char_wb(3-5)',
    lambda: MultinomialNB(alpha=0.5),
)
pd.DataFrame(tuning_runs).tail(9)

In [ ]:
for n_estimators, max_depth in [(200, None), (400, 40), (300, 25)]:
    depth_label = 'None' if max_depth is None else str(max_depth)
    log_classical_run(
        'Random Forest', 'svd', 'n_estimators=' + str(n_estimators) + ', max_depth=' + depth_label,
        lambda n_estimators=n_estimators, max_depth=max_depth: RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=SEED),
    )
log_classical_run(
    'Random Forest', 'w2v', 'n_estimators=300, max_depth=None',
    lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
)
pd.DataFrame(tuning_runs).tail(4)

## 8. RNN Family in PyTorch *(spec §3.7 — SimpleRNN, GRU, LSTM + bidirectional variants)*

One shared vocabulary, one builder class — the comparison isolates the recurrent cell and directionality. Each of the 6 variants is trained with **3 from-scratch configurations** (hidden size / dropout / learning rate) **plus a 4th GloVe-initialized configuration** (round-2 optimization — embedding layer starts from pretrained `glove-wiki-gigaword-100` vectors, fine-tuned at lr/10; see the round-2 change log after §11.1), early stopping on validation accuracy (patience 2, max 8 epochs).

**Readout = masked mean pooling (design decision).** Sequences are post-padded to `MAX_LEN=150`, but the median question has ~19 words (p95 ≈ 106), so a naive `out[:, -1, :]` readout reads a state that has iterated through ~110 zero-input padding steps *after* the last real word. For an un-gated SimpleRNN that is fatal: with no input, `h_t = tanh(W_hh·h_{t-1} + b)` contracts monotonically toward an input-independent fixed point, so the classifier sees nearly the same vector for every class — observed empirically as SimpleRNN pinned at the 10% class prior (`val_f1 ≈ 0.02`, early stop in every configuration) while gated GRU/LSTM coped via their additive / constant-error-carousel state updates. The fix pools output states over non-padding positions only (`mask = x != 0`), which preserves every word's evidence *and* shortens each loss-to-word BPTT path. Residual caveat: the backward direction of Bi- variants still consumes the padding first — `pack_padded_sequence` would remove that (noted in §12). A small purge cell below keeps `tuning_runs` free of duplicate RNN rows when this section is re-run.

In [ ]:
from collections import Counter

MAX_TOKENS = 30000
MAX_LEN = 150

counter = Counter(tok for doc in tokenized_train for tok in doc)
itos = ['<pad>', '<unk>'] + [w for w, _ in counter.most_common(MAX_TOKENS - 2)]
stoi = {w: i for i, w in enumerate(itos)}

def encode_tokens(tokens):
    ids = [stoi.get(t, 1) for t in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

X_train_seq = torch.tensor([encode_tokens(d) for d in tokenized_train], dtype=torch.long)
X_val_seq = torch.tensor([encode_tokens(d) for d in tokenized_val], dtype=torch.long)
X_test_seq = torch.tensor([encode_tokens(d) for d in tokenized_test], dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print('sequences:', tuple(X_train_seq.shape), '| vocabulary:', len(itos),
      f'| MAX_LEN={MAX_LEN} covers p{int(100 * (train_part["n_words"] <= MAX_LEN).mean())} of train questions')

In [ ]:
from tqdm.auto import tqdm

# --- Round 2 (post-first-run optimization): GloVe-initialized embeddings ----------
# First full run: all 6 RNN variants landed at 0.588-0.600 test macro-F1, BELOW
# the classical TF-IDF models (0.64-0.65). Diagnosis: a from-scratch 128-d
# embedding layer must learn what words MEAN out of only 40k noisy questions,
# while TF-IDF hands the classifier 35k-116k informative lexical features from
# step one. Fix: initialize the embedding layer with pretrained
# glove-wiki-gigaword-100 vectors (already cached from section 6) so training
# only has to adapt sequence structure and the classifier head.
# NOTE: the section-6 cell rebinds `glove` to stacked sentence matrices, so the
# KeyedVectors are re-loaded here from the warm gensim cache (~20 s, no download).
glove_kv = gensim.downloader.load('glove-wiki-gigaword-100')
_glove_rng = np.random.default_rng(SEED)   # seeded -> reproducible OOV init
GLOVE_EMBEDDING_MATRIX = np.zeros((len(itos), 100), dtype=np.float32)
_found = 0
for _idx, _word in enumerate(itos):
    if _idx < 2:                            # <pad>, <unk> stay all-zero
        continue
    if _word in glove_kv:
        GLOVE_EMBEDDING_MATRIX[_idx] = glove_kv[_word]
        _found += 1
    else:                                   # typos / rare tokens: small random vectors
        GLOVE_EMBEDDING_MATRIX[_idx] = _glove_rng.normal(0.0, 0.1, 100)
print(f'GloVe init: {_found}/{len(itos) - 2} vocab types covered '
      f'({_found / (len(itos) - 2):.1%}); OOV types get seeded N(0, 0.1) vectors')


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden, cell_type, bidirectional, num_classes, dropout,
                 embedding_matrix=None, freeze_embeddings=False):
        super().__init__()
        if embedding_matrix is not None:
            # Round 2: start from pretrained GloVe vectors instead of random noise
            # (transfer learning); embed_dim follows the matrix (100 for glove-100).
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(embedding_matrix, dtype=torch.float32),
                freeze=freeze_embeddings, padding_idx=0)
            embed_dim = embedding_matrix.shape[1]
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = cell_type(embed_dim, hidden, batch_first=True, bidirectional=bidirectional)
        self.directions = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * self.directions, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out = self.rnn(emb)[0]                       # (batch, seq, hidden * dirs)
        # Masked mean pooling over real-token positions (padding id = 0). A plain
        # last-timestep readout is destroyed by post-padding: with ~110 zero-input
        # steps after the words, an un-gated SimpleRNN contracts to an
        # input-independent fixed point, pinning it at the 10% class prior.
        # Pooling keeps every word's state and shortens each BPTT path.
        mask = (x != 0).unsqueeze(-1).to(out.dtype)  # (batch, seq, 1)
        summed = (out * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)       # (batch, 1); guards empty text
        pooled = summed / lengths
        return self.fc(self.dropout(pooled))

def make_loader(X, y, batch_size, shuffle):
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

def evaluate_torch(model, X, y_t, batch_size=256):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in tqdm(make_loader(X, y_t, batch_size, False),
                         desc='  eval', leave=False):
            preds.append(model(xb.to(DEVICE)).argmax(dim=1).cpu().numpy())
    preds = np.concatenate(preds)
    y_np = y_t.numpy()
    return accuracy_score(y_np, preds), f1_score(y_np, preds, average='macro'), preds

def train_eval_rnn(cell_type, bidirectional, hidden, dropout, lr, epochs=8, batch_size=64,
                   embedding_matrix=None, freeze_embeddings=False):
    torch.manual_seed(SEED)
    model = RNNClassifier(len(itos), 128, hidden, cell_type, bidirectional, len(CLASS_NAMES), dropout,
                          embedding_matrix=embedding_matrix,
                          freeze_embeddings=freeze_embeddings).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    if embedding_matrix is not None and not freeze_embeddings:
        # Fine-tune pretrained embeddings 10x slower than the rest of the network:
        # Adam at 1e-3 would wash out the GloVe geometry within the first epochs
        # (catastrophic forgetting), defeating the point of pretraining.
        optimizer = torch.optim.Adam([
            {'params': model.embedding.parameters(), 'lr': lr / 10},
            {'params': [p for n, p in model.named_parameters() if not n.startswith('embedding.')]},
        ], lr=lr)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_acc, best_state = 0.0, None
    patience_left = 2
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in tqdm(make_loader(X_train_seq, y_train_t, batch_size, True),
                           desc=f'  epoch {epoch}/{epochs}', leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        acc, f1m, _ = evaluate_torch(model, X_val_seq, y_val_t)
        print(f'  epoch {epoch}: train_loss={total:.3f} val_acc={acc:.4f} val_f1={f1m:.4f}')
        if acc > best_acc:
            best_acc, patience_left = acc, 2
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_left -= 1
            if patience_left == 0:
                print('  early stop')
                break
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model

print('RNN components ready')

In [ ]:
# Re-running the RNN section (e.g. after the masked-pooling readout fix, or a
# hot-patch mid-session): drop this section's earlier rows from the tuning log
# so re-execution appends fresh rows instead of duplicating them. On a clean
# full run this is a no-op - at this point only classical rows exist.
RNN_MODEL_NAMES = ('SimpleRNN', 'GRU', 'LSTM', 'Bi-SimpleRNN', 'Bi-GRU', 'Bi-LSTM')
tuning_runs = [r for r in tuning_runs if r['model'] not in RNN_MODEL_NAMES]
print('tuning rows kept:', len(tuning_runs))

In [ ]:
RNN_VARIANTS = {
    'SimpleRNN': (nn.RNN, False),
    'GRU': (nn.GRU, False),
    'LSTM': (nn.LSTM, False),
    'Bi-SimpleRNN': (nn.RNN, True),
    'Bi-GRU': (nn.GRU, True),
    'Bi-LSTM': (nn.LSTM, True),
}
RNN_CONFIGS = [
    dict(hidden=64, dropout=0.3, lr=1e-3),
    dict(hidden=128, dropout=0.3, lr=1e-3),
    dict(hidden=64, dropout=0.5, lr=3e-4),
    # Round 2 (accuracy optimization): the best from-scratch config retrained
    # with the embedding layer initialized from pretrained GloVe 100-d vectors
    # (matrix built in the components cell above), fine-tuned at lr/10. Added
    # as a 4th config so the from-scratch vs pretrained ablation stays visible
    # in the section-10 tuning table (spec minimum of 3 configs per model kept).
    # Optional frozen variant: add freeze=True to the dict to lock the vectors.
    dict(hidden=128, dropout=0.3, lr=1e-3, embed='glove'),
]
rnn_models = {}
for name, (cell_type, bidirectional) in RNN_VARIANTS.items():
    print('===', name, '===')
    best_f1, best_model, best_label = -1.0, None, ''
    for cfg in RNN_CONFIGS:
        config_label = 'hidden=' + str(cfg['hidden']) + ', dropout=' + str(cfg['dropout']) + ', lr=' + str(cfg['lr'])
        matrix = GLOVE_EMBEDDING_MATRIX if cfg.get('embed') == 'glove' else None
        if matrix is not None:
            config_label += ', embed=glove100-' + ('frozen' if cfg.get('freeze') else 'ft')
        print(' config:', config_label)
        model = train_eval_rnn(cell_type, bidirectional,
                               hidden=cfg['hidden'], dropout=cfg['dropout'], lr=cfg['lr'],
                               embedding_matrix=matrix, freeze_embeddings=bool(cfg.get('freeze', False)))
        acc, f1m, _ = evaluate_torch(model, X_val_seq, y_val_t)
        tuning_runs.append({
            'model': name,
            'representation': 'glove-init' if matrix is not None else 'torch-embedding',
            'config': config_label,
            'val_accuracy': acc, 'val_macro_f1': f1m,
        })
        if f1m > best_f1:
            best_f1, best_model, best_label = f1m, model, config_label
    rnn_models[name] = best_model
    print(name, 'best:', best_label, '-> val_macro_f1 =', round(best_f1, 4))
print('RNN tuning complete')

## 9. BERT Base *(spec §3.7)*

`bert-base-uncased`, max length 96 (justified by the §3 length percentiles and 12GB VRAM), classifier head with 10 labels. **3 configurations** (lr / batch size / epochs), best checkpoint by validation macro-F1 kept in memory for the final test evaluation. On the B580 this section takes ≈ 1.5–2.5 h; on CPU it is impractical — switch to a GPU/XPU runtime or reduce `N_PER_CLASS_*` before attempting it.

In [ ]:
from tqdm.auto import tqdm

from transformers import AutoTokenizer, BertForSequenceClassification

BERT_NAME = 'bert-base-uncased'
BERT_MAX_LEN = 96
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)

def bert_tensors(texts):
    enc = bert_tokenizer(list(texts), truncation=True, padding='max_length',
                         max_length=BERT_MAX_LEN, return_tensors='pt')
    return enc['input_ids'], enc['attention_mask'], enc['token_type_ids']

bert_train = bert_tensors(X_train_text)
bert_val = bert_tensors(X_val_text)
bert_test = bert_tensors(X_test_text)
print('BERT tensors:', tuple(bert_train[0].shape), tuple(bert_val[0].shape), tuple(bert_test[0].shape))

def bert_loader(tensors, y_t, batch_size, shuffle):
    return DataLoader(TensorDataset(*tensors, y_t), batch_size=batch_size, shuffle=shuffle)

def evaluate_bert(model, tensors, y_np, batch_size=128):
    model.eval()
    preds = []
    with torch.no_grad():
        for ids, mask, tt, _ in tqdm(bert_loader(tensors, torch.tensor(y_np), batch_size, False),
                                     desc='  eval', leave=False):
            logits = model(ids.to(DEVICE), attention_mask=mask.to(DEVICE), token_type_ids=tt.to(DEVICE)).logits
            preds.append(logits.argmax(dim=1).cpu().numpy())
    preds = np.concatenate(preds)
    return accuracy_score(y_np, preds), f1_score(y_np, preds, average='macro'), preds

def train_eval_bert(lr, batch_size, epochs):
    torch.manual_seed(SEED)
    model = BertForSequenceClassification.from_pretrained(BERT_NAME, num_labels=len(CLASS_NAMES)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best = {'f1': -1.0, 'state': None}
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for ids, mask, tt, yb in tqdm(bert_loader(bert_train, y_train_t, batch_size, True),
                                      desc=f'  epoch {epoch}/{epochs}', leave=False):
            ids, mask, tt, yb = ids.to(DEVICE), mask.to(DEVICE), tt.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(ids, attention_mask=mask, token_type_ids=tt).logits, yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        acc, f1m, _ = evaluate_bert(model, bert_val, y_val)
        print(f'  epoch {epoch}: train_loss={total:.3f} val_acc={acc:.4f} val_f1={f1m:.4f}')
        if f1m > best['f1']:
            best = {'f1': f1m,
                    'state': {k: v.cpu().clone() for k, v in model.state_dict().items()}}
    model.load_state_dict(best['state'])
    model.to(DEVICE)
    return model, best['f1']

print('BERT components ready')

In [ ]:
BERT_CONFIGS = [
    dict(lr=2e-5, batch_size=16, epochs=2),
    dict(lr=3e-5, batch_size=16, epochs=2),
    dict(lr=2e-5, batch_size=32, epochs=3),
]
best_bert_model, best_bert_f1 = None, -1.0
for cfg in BERT_CONFIGS:
    label = 'lr=' + str(cfg['lr']) + ', batch=' + str(cfg['batch_size']) + ', epochs=' + str(cfg['epochs'])
    print('=== BERT:', label, '===')
    model, val_f1 = train_eval_bert(**cfg)
    acc, f1m, _ = evaluate_bert(model, bert_val, y_val)
    tuning_runs.append({
        'model': 'BERT Base', 'representation': 'bert-uncased', 'config': label,
        'val_accuracy': acc, 'val_macro_f1': f1m,
    })
    if f1m > best_bert_f1:
        best_bert_f1, best_bert_model = f1m, model
print('BERT tuning complete | best val_macro_f1 =', round(best_bert_f1, 4))

## 10. Hyperparameter Tuning Log *(spec §3.8)*

The dedicated, consolidated table of **all** tuning runs (22 classical + 24 RNN — 18 from-scratch + 6 GloVe-initialized — + 3 BERT = 49 runs; 43 before the round-2 GloVe configs). The best row per model by validation macro-F1 is carried into the final test evaluation.

In [ ]:
tuning_table = pd.DataFrame(tuning_runs)
tuning_table.to_csv(os.path.join(OUT_DIR, 'tuning_runs.csv'), index=False)
display(tuning_table)

best_configs = tuning_table.loc[tuning_table.groupby('model')['val_macro_f1'].idxmax()].reset_index(drop=True)
display(best_configs[['model', 'representation', 'config', 'val_accuracy', 'val_macro_f1']])

## 11. Final Evaluation on the Test Set *(spec §3.9)*

Each model's **best configuration** is evaluated once on the untouched official-test subsample: accuracy, macro-F1, full classification report, confusion matrix. Classical winners are refit on train; RNN/BERT winners reuse the checkpoints stored above.

In [ ]:
final_results = []
test_predictions = {}

CLASSICAL_NAMES = ['Logistic Regression', 'Naive Bayes', 'Random Forest']
for _, row in best_configs[best_configs['model'].isin(CLASSICAL_NAMES)].iterrows():
    spec = next(s for s in classical_specs
                if s['model'] == row['model'] and s['config'] == row['config']
                and s['representation'] == row['representation'])
    X_tr, _, X_te = REPS[spec['representation']]
    fitted = spec['factory']().fit(X_tr, y_train)
    pred = fitted.predict(X_te)
    test_predictions[row['model']] = pred
    final_results.append({
        'model': row['model'],
        'test_accuracy': accuracy_score(y_test, pred),
        'test_macro_f1': f1_score(y_test, pred, average='macro'),
    })
    print(row['model'], 'done')
pd.DataFrame(final_results)

In [ ]:
for name, model in rnn_models.items():
    _, f1m, pred = evaluate_torch(model, X_test_seq, y_test_t)
    test_predictions[name] = pred
    final_results.append({
        'model': name,
        'test_accuracy': accuracy_score(y_test, pred),
        'test_macro_f1': f1m,
    })
    print(name, 'done')
pd.DataFrame(final_results)

In [ ]:
if best_bert_model is not None:
    _, f1m, pred = evaluate_bert(best_bert_model, bert_test, y_test)
    test_predictions['BERT Base'] = pred
    final_results.append({
        'model': 'BERT Base',
        'test_accuracy': accuracy_score(y_test, pred),
        'test_macro_f1': f1m,
    })
    print('BERT Base done')
results_table = pd.DataFrame(final_results).sort_values('test_macro_f1', ascending=False).reset_index(drop=True)
results_table.to_csv(os.path.join(OUT_DIR, 'final_results.csv'), index=False)
results_table

In [ ]:
plot_table = results_table.melt(id_vars='model', value_vars=['test_accuracy', 'test_macro_f1'],
                                var_name='metric', value_name='score')
plt.figure(figsize=(13, 5))
sns.barplot(data=plot_table, x='model', y='score', hue='metric')
plt.xticks(rotation=40, ha='right')
plt.ylim(0, 1)
plt.title('Test-set performance across all models (best configuration each)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'model_comparison.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
top_models = results_table['model'].head(4).tolist()
if 'BERT Base' in test_predictions and 'BERT Base' not in top_models:
    top_models.append('BERT Base')

fig, axes = plt.subplots(1, len(top_models), figsize=(4.2 * len(top_models), 4))
for ax, name in zip(np.atleast_1d(axes), top_models):
    matrix = confusion_matrix(y_test, test_predictions[name], normalize='true')
    sns.heatmap(matrix, annot=False, cmap='Blues', cbar=True,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(name)
    ax.tick_params(axis='x', rotation=90)
    ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'confusion_matrices.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
best_name = results_table.iloc[0]['model']
worst_name = results_table.iloc[-1]['model']
print('BEST MODEL:', best_name)
print(classification_report(y_test, test_predictions[best_name], target_names=CLASS_NAMES))
print('=' * 70)
print('WORST MODEL:', worst_name)
print(classification_report(y_test, test_predictions[worst_name], target_names=CLASS_NAMES))

### 11.1 Comparison and Discussion

Write the analysis against the tables/figures above (numbers come from your run — do not copy generic values into the report):
- Best- and worst-performing model, by both accuracy and macro-F1, with the size of the gap.
- Why the winners won — connect to the EDA (question lengths, noise, per-topic vocabulary, GloVe OOV on misspellings) and to theory: sparse high-dimensional TF-IDF + linear models vs dense mean-pooled non-contextual embeddings vs contextual self-attention; gating and bidirectional context in GRU/LSTM vs the vanishing-gradient behaviour of SimpleRNN; last-timestep readout vs masked pooling.
- **RNN readout ablation (strong report material):** the last-timestep readout pins SimpleRNN at the class prior because ~110 zero-input padding steps contract an un-gated state to an input-independent fixed point (best `val_macro_f1 ≈ 0.03` pre-fix, early stop in every config); masked mean pooling recovers it while gated cells (GRU/LSTM) cope either way. Isolate this effect when comparing SimpleRNN vs GRU/LSTM in the table — the remaining gap after the fix is attributable to gating, not to the readout.
- Which topics confuse with which (read the heatmaps — expected candidates: Politics & Government ↔ Society & Culture, Science & Mathematics ↔ Health, Business & Finance ↔ Politics & Government) and what that says about category semantics.
- Whether fine-tuned BERT justifies its training cost over the classical baselines on noisy UGC; where the selected preprocessing strategy helped or hurt.

# **AFIF** 

* **Narrative for Your Lab Report:** This experiment will let you draw a definitive 3-tier hierarchy in your report discussion: $$ \text{From-Scratch RNN (59.6\%)} \longrightarrow \text{Pretrained GloVe RNN (63--65\%)} \longrightarrow \text{Contextual BERT (70.2\%)} $$ This clearly explains how lexical TF-IDF, static word embeddings, and bidirectional self-attention compare under noisy user-generated text constraints.*

## Key findings After 1st full run

*(configs corrected to match the actual §10/§11 outputs of this run)*


### 1. Final Test-Set Performance Leaderboard (Macro-F1 & Accuracy)

| Rank | Model Family | Best Configuration | Test Accuracy | Test Macro-F1 | Notes / Architecture Behavior |
| :--- | :--- | :--- | :---: | :---: | :--- |
| 🥇 **1** | **BERT Base** | `lr=2e-5, batch=32, epochs=3` | **70.19%** | **0.6977** | **Clear Winner.** Strongest semantic contextual understanding across all 10 topics (best val 0.7021 at epoch 2; epoch 3 dipped, best-checkpoint restore kept epoch 2). |
| 🥈 **2** | **Logistic Regression** | `C=1.0, char_wb(3-5) n-grams` | **65.11%** | **0.6490** | **Best Classical ML.** The char n-gram representation beat word TF-IDF (best word run: C=2.0, val 0.6454) — misspelling-robustness hypothesis confirmed. |
| 🥉 **3** | **Naive Bayes** | `MultinomialNB alpha=0.3` (on TF-IDF) | **64.65%** | **0.6428** | Strong probabilistic baseline; close to Logistic Regression. |
| **4** | **Random Forest** | `n_estimators=300, max_depth=None` (on W2V) | **60.77%** | **0.6037** | Word2Vec mean embeddings outperformed 200-d SVD/LSA (`49.7%`). |
| **5** | **Bi-LSTM** | `hidden=64, dropout=0.3, lr=1e-3` | **59.61%** | **0.5963** | **Best RNN.** Bidirectional cell states effectively capture forward & backward context. |
| **6** | **Bi-GRU** | `hidden=64, dropout=0.3, lr=1e-3` | **59.43%** | **0.5923** | Gated architecture closely matching Bi-LSTM. |
| **7** | **LSTM** | `hidden=128, dropout=0.3, lr=1e-3` | **59.34%** | **0.5912** | Unidirectional gated cell. |
| **8** | **SimpleRNN** | `hidden=128, dropout=0.3, lr=1e-3` | **59.31%** | **0.5906** | Recovered from 10% to ~59% after the **masked-pooling readout** fix. |
| **9** | **Bi-SimpleRNN** | `hidden=128, dropout=0.3, lr=1e-3` | **59.15%** | **0.5888** | Similar to SimpleRNN without gating. |
| **10** | **GRU** | `hidden=128, dropout=0.3, lr=1e-3` | **58.78%** | **0.5843** | Unidirectional GRU — last place by only 0.5 pt. |

### 2. Key Findings & Insights for Your Lab Report (§11)

1. **The Representation Gap (BERT vs. Bag-of-Words vs. RNN Embeddings)**:
   - **BERT Base achieved a ~5.0% accuracy advantage over the best classical baseline** (70.19% vs. 65.11%) and **~10.6% over the best RNN** (70.19% vs. 59.61%).
   - *Why BERT won*: Pretrained bidirectional self-attention captures polysemy, syntactic nuance, and noisy user slang (WordPiece tokenization handles misspellings where static embeddings fail).

2. **Classical ML (TF-IDF / char n-grams) Beats RNNs from Scratch**:
   - Both **Logistic Regression (65.11%)** and **Multinomial Naive Bayes (64.65%)** outperformed all 6 RNN variants (~59%).
   - *Why*: In topic classification, distinctive unigrams/bigrams (*"GPU"*, *"stocks"*, *"pediatrician"*) carry very strong direct signals. Sparse features give linear models direct access to 35k word-TF-IDF / 116k char n-gram lexical features, whereas RNNs trained from scratch on 40k questions must learn both token embeddings and temporal transitions simultaneously. *(Round 2 addresses exactly this — GloVe-initialized embeddings; see the change log below.)*

3. **RNN Family Dynamics & The Masked-Pooling Impact**:
   - The un-gated **SimpleRNN was pinned at the 10% class prior** (`val_f1 ≈ 0.02`) by the last-timestep readout iterating through ~110 post-padding zero-input steps, while gated GRU/LSTM coped via their additive state updates.
   - Introducing **Masked Average Pooling** over non-padding tokens recovered SimpleRNN from **10% to ~59.3%**.
   - **Bi-LSTM tops the family, but all six variants sit within ~1.2 pts** — with masked pooling, gating and direction effects are small at this data scale; the binding constraint is the from-scratch embedding layer (see Round 2 changes).

4. **Confusion Patterns (Semantic Topic Overlap)**:
   - The confusion matrices show that top confusions occur between semantically related domains:
     - **Politics & Government ↔ Society & Culture**
     - **Science & Mathematics ↔ Health**
     - **Business & Finance ↔ Politics & Government**
   - Distinctive topics like **Sports** (0.86 F1) and **Computers & Internet** (0.83) achieve the highest class F1-scores; **Education & Reference** (0.50) and **Business & Finance** (0.54) are weakest — these categories genuinely overlap others.

# Round 2 Results — After the 2nd Full Run (GloVe-Initialized RNNs)

**What changed:** every RNN variant gained a 4th config — `hidden=128, dropout=0.3, lr=1e-3, embed=glove100-ft` — where the embedding layer starts from pretrained `glove-wiki-gigaword-100` vectors instead of random noise, fine-tuned at lr/10 (see the Round 2 change log below). Everything else in the notebook is unchanged; the whole notebook was re-run fresh.

**GloVe coverage:** 20,477 / 29,998 vocab types (**68.3%**); the uncovered third is mostly misspellings and hapax tokens. 99.7% of training questions contain at least one known token (§6), so effective coverage is far higher than the type-level 68.3%.

### 1. Effect of GloVe Initialization — From-Scratch vs Pretrained (per variant)

| Variant | Scratch best (val macro-F1) | GloVe-init (val macro-F1) | Δ val | Test macro-F1: run 1 → run 2 |
| :--- | :---: | :---: | :---: | :---: |
| SimpleRNN | 0.597 | 0.626 | **+2.9** | 0.591 → 0.620 |
| Bi-SimpleRNN | 0.595 | 0.632 | **+3.6** | 0.589 → 0.630 |
| Bi-LSTM | 0.600 | 0.659 | **+5.9** | 0.596 → 0.655 |
| LSTM | 0.589 | 0.652 | **+6.3** | 0.591 → 0.644 |
| Bi-GRU | 0.594 | **0.662** | **+6.8** | 0.592 → **0.659** |
| GRU | 0.584 | 0.654 | **+7.0** | 0.584 → 0.648 |

Average gain **+5.4 pts** validation macro-F1 — every single variant improved, and the GloVe config is the new best config for **all six** variants (§10 best-configs table).

### 2. Updated Test-Set Leaderboard (run 2, 10k official test)

| Rank | Model | Best Configuration | Test Acc | Test Macro-F1 | Movement vs run 1 |
| :--- | :--- | :--- | :---: | :---: | :--- |
| 🥇 **1** | **BERT Base** | `lr=2e-5, batch=32, epochs=3` | **70.02%** | **0.6960** | reproduced within ±0.2 pt (XPU non-determinism) |
| 🥈 **2** | **Bi-GRU** | `h=128, dropout=0.3, lr=1e-3, embed=glove100-ft` | **66.13%** | **0.6589** | ⬆ 0.592 → 0.659 — **now beats the best classical model by +1.0** |
| 🥉 **3** | **Bi-LSTM** | `h=128, dropout=0.3, lr=1e-3, embed=glove100-ft` | **65.63%** | **0.6549** | ⬆ 0.596 → 0.655 — beats LogReg by +0.6 |
| 4 | Logistic Regression | `C=1.0, char_wb(3-5)` | 65.11% | 0.6490 | unchanged |
| 5 | GRU | `h=128, ..., embed=glove100-ft` | 65.05% | 0.6484 | ⬆ 0.584 → 0.648 — ties LogReg |
| 6 | LSTM | `h=128, ..., embed=glove100-ft` | 64.73% | 0.6441 | ⬆ 0.591 → 0.644 |
| 7 | Naive Bayes | `MultinomialNB alpha=0.3` | 64.65% | 0.6428 | unchanged |
| 8 | Bi-SimpleRNN | `h=128, ..., embed=glove100-ft` | 63.37% | 0.6296 | ⬆ 0.589 → 0.630 |
| 9 | SimpleRNN | `h=128, ..., embed=glove100-ft` | 62.53% | 0.6200 | ⬆ 0.591 → 0.620 |
| 10 | Random Forest | `n_estimators=300` (on W2V) | 60.72% | 0.6031 | **now last place** — every RNN overtakes it |

### 3. Key Findings (run 2)

1. **Representation was the binding constraint, not recurrence.** In run 1 all six variants sat within 1.2 pts — architecture didn't matter because from-scratch embeddings bottlenecked everything. With GloVe, the spread opens to 3.9 pts and the textbook ordering finally appears: **Bi-GRU > Bi-LSTM > GRU > LSTM > Bi-SimpleRNN > SimpleRNN**. Gating and bidirectionality only pay off once words carry meaningful vectors. Combined story across both runs: fix the readout (run 1) → fix the representation (run 2) → architecture effects emerge.
2. **Neural networks now beat the classical ceiling.** Bi-GRU (+1.0) and Bi-LSTM (+0.6) overtake LogReg+char; every RNN variant overtakes Random Forest (which drops to last). Run 1's "classical beats RNNs" narrative is overturned — it was a symptom of under-trained embeddings, not a property of RNNs.
3. **BERT's lead shrank from 10.1 to 3.7 pts** macro-F1 (0.6960 vs 0.6589). Static pretrained features close most of the gap; contextual pretraining still wins. Final 3-tier hierarchy: **from-scratch RNN ≈ 0.59 → GloVe RNN ≈ 0.62–0.66 → BERT ≈ 0.70**.
4. **Gains scale with architectural capacity:** un-gated SimpleRNN gains least (+2.9), gated/bidirectional variants most (+5.9 to +7.0) — better inputs only help models equipped to exploit them (gates preserve information over longer evidence spans).

### 4. Training Dynamics (evidence the lr/10 guard worked)

- **Bi-GRU's GloVe config reached 0.633 val macro-F1 in epoch 1** — above every scratch config's *final* 8-epoch best (max 0.600). The pretrained geometry is useful from the first gradient step.
- All six GloVe runs climb smoothly and monotonically with **no early stops** and no epoch-1 collapse — fine-tuning embeddings at lr/10 preserved the pretrained structure instead of washing it out (catastrophic forgetting avoided).
- GloVe runs were still creeping up at epoch 8 (e.g. GRU: 0.6367 → 0.6537 across epochs 3–8), suggesting 10–12 epochs could add ~0.3–0.5 pt more.

### 5. Reproducibility Note

This was a complete fresh re-run (not a hot-patch): deterministic classical rows (LogReg, NB-tfidf) reproduced exactly; BERT and neural runs vary within ±0.2 pt across runs (GPU/XPU non-determinism). Run-1 vs run-2 differences are therefore far larger than run-to-run noise for every claimed effect.

## Three findings worth building the report around
1. **The char representation won classical** — LogReg+char 0.657 val beats word TF-IDF (0.645) and everything else, exactly as the misspelling-robustness hypothesis predicted. Your changelog table is now fully validated by the completed run.
2. **Post-fix, gating and bidirectionality barely matter.** All six RNN variants sit in a 1.2-pt band — and SimpleRNN beats GRU on test. With masked pooling, the classic "GRU/LSTM > SimpleRNN" story collapses at this scale: the RNNs are limited by their from-scratch 128-d embeddings, not by gradient flow. (Note: my changelog's "expected to trail GRU/LSTM" line should be updated to this — see below.)
3. **RNNs (0.59) < classical (0.65) < BERT (0.70).** 116k sparse char features carry more signal than an embedding layer learned on 40k noisy questions; contextual pretraining adds +5 pts on top. Per-class, BERT is excellent on Sports (0.86 F1) and Computers (0.83), weak on Education & Reference (0.50) and Business & Finance (0.54) — those classes semantically overlap others, good heatmap material.

# Round 2 — changes made after the 1st full run (accuracy optimization)

**Where we stood after run 1** (40k/5k/10k subsample, Arc B580): BERT 0.702/0.698 · LogReg+char 0.651/0.649 · NB 0.647/0.643 · RF 0.608/0.604 · **all six RNN variants 0.588–0.596 test macro-F1 — the whole RNN family sat ~5 pts *below* the classical models.** The numbers in the leaderboard above are run-1 results; they refresh automatically after the re-run below.

**Diagnosis.** The RNNs had to learn a 128-d embedding layer from scratch on only 40k noisy questions, so most topic words (seen 2–5 times) never reached useful vectors — while the TF-IDF models start from 35k–116k informative lexical features on step one.

| # | Change | Where | Why |
|---|---|---|---|
| 1 | **GloVe-initialized embeddings** — a `(30000, 100)` matrix from `glove-wiki-gigaword-100`: in-vocab words → pretrained vector, `<pad>`/`<unk>` → zeros, OOV (typos, rare tokens) → seeded `N(0, 0.1)`; injected via `nn.Embedding.from_pretrained(...)`, masked mean-pooling readout unchanged | §8 components cell | transfer learning — the RNN spends its capacity on sequence structure and classification instead of re-learning English vocabulary |
| 2 | **Safe fine-tuning schedule** — embeddings train at `lr/10` via optimizer param-groups (`freeze_embeddings=` also wired for a frozen variant) | `train_eval_rnn` | Adam at 1e-3 would wash out the GloVe geometry within the first epochs (catastrophic forgetting), defeating the point of pretraining |
| 3 | **Added as a 4th config, not a replacement** — `hidden=128, dropout=0.3, lr=1e-3, embed=glove100-ft` per variant, on top of the 3 scratch configs | §8 training cell | keeps the from-scratch vs pretrained ablation readable in the §10 table; spec minimum of ≥3 configs per model still satisfied |

- Tuning table grows **43 → 49 rows** (18 scratch + 6 GloVe-initialized RNN runs); §10/§11 refresh automatically.
- **Expected:** +3–5 pts macro-F1 for the RNN family (~0.62–0.645) — parity with NB, near LogReg+char. GloVe cannot see typo/morphology signal the way char n-grams do, and BERT (contextual) should stay ahead. Either outcome strengthens the report narrative: *static pretrained features close most of the gap to sparse lexical models; contextual pretraining still wins.*
- **Cost:** ~20–40 min extra training on the B580; no new downloads (GloVe cache already warm from §6).
- **To apply (in a live kernel or fresh run):** re-run the §8 components cell → the purge cell → the training cell, then §10–§11. The purge cell drops old RNN rows first, so re-running cannot duplicate rows.

# Bert Accuracy Improvement
Here is an honest, technical, and strategic assessment of the proposed BERT optimization plan:

---

### 1. The Core Diagnosis (Item #1: Natural Text vs. Lemmatized) — **100% Accurate & Critical**

- **Why this is a real flaw:** 
  In your current notebook, `bert_train = bert_tensors(X_train_text)` receives `X_train_text`, which was cleaned using `SELECTED_STRATEGY` (**lemmatized**).
- **Why it hurts BERT:**
  - `MultinomialNB` and `TF-IDF` love lemmatization because collapsing *"running"*, *"ran"*, and *"runs"* into *"run"* reduces matrix sparsity.
  - **Transformers (BERT) hate lemmatization.** BERT's WordPiece tokenizer (`bert-base-uncased`) was pretrained on natural grammatical English. BERT relies heavily on auxiliary verbs (*"is"*, *"would"*, *"have"*), punctuation (*"?"*, *"!"*), and morphological affixes (*"-ing"*, *"-ed"*) to build attention vectors. 
  - Stripping stopwords and lemmatizing text essentially turns natural sentences into unnatural "keyword soup", degrading BERT's attention mechanism.
- **Verdict on #1:** **Must fix immediately.** It is a zero-compute change with an immediate **+1.0 to +1.5 pt gain**.

---

### 2. Assessment of the Full Menu (Items #2 through #8)

| # | Proposed Change | Practical Value | Impact & Evaluation |
| :---: | :--- | :---: | :--- |
| **#1** | **Raw/Minimal text for BERT** | 🟢 **Essential** | Zero-cost fix; restores BERT's native WordPiece input format. |
| **#3** | **`text_pair` encoding (`[CLS] Title [SEP] Body [SEP]`)** | 🟢 **High** | Uses BERT's native segment architecture (`token_type_ids = 0` for title, `1` for content). Zero-cost. |
| **#4** | **`max_len` 96 $\to$ 128** | 🟢 **High** | 96 was chosen to fit low VRAM. On your **12GB Arc B580**, `max_len=128` with `batch_size=16` easily fits into ~5–6GB VRAM. Prevents truncating longer question bodies. |
| **#5** | **LR Scheduler (Linear Warmup + Cosine/Linear Decay) + Weight Decay (0.01) + Grad Clip (1.0)** | 🟢 **Essential** | Standard Hugging Face best practice (`get_linear_schedule_with_warmup`). Fixes the epoch-3 performance dip caused by fixed-rate AdamW. |
| **#8** | **`torch.autocast('xpu', torch.bfloat16)`** | 🟢 **High** | Intel Arc B580 has dedicated XMX AI engines with native hardware `bfloat16`. Enables **~1.5x–2x faster training** and halves VRAM consumption. |
| **#7** | **Cross-Family Ensemble (BERT + Bi-GRU + LogReg-char)** | 🟢 **High (Bonus)** | Directly fulfills §12 ("Future work / bonus: ensemble of top models"). Fuses three fundamentally different error distributions (Self-Attention + Recurrent Gating + Subword Lexical). |
| **#2** | **Scale data (40k $\to$ 120k)** | 🟡 **Optional / Heavy** | Will give +2–3%, but will triple runtimes across the whole notebook. Only do this if you want absolute peak leaderboard numbers. |
| **#6** | **3-Seed BERT Ensemble** | 🟡 **Low Priority** | Triples BERT training time for a marginal gain (+0.5%). Cross-family voting (#7) is much more interesting for your report. |

---

### 3. Impact on Your Academic Spec & Report

- **Course Spec Compliance**: The changes preserve the required multi-config tuning log (§3.8) and keep `bert-base-uncased` within the official course constraints.
- **Research Narrative**: You can write a compelling comparative section in §11:
  > *"Why Preprocessing Depth Depends on Model Architecture: While classical Bag-of-Words and Naive Bayes benefit from aggressive lemmatization to combat feature sparsity, contextual pretrained transformers perform best on natural, unmodified text where syntactic affixes and stopwords carry critical attention signal."*


## 12. Conclusion *(spec §5.2)*

Key takeaways, limitations, and future work — fill from your results:
- **Limitations:** stratified subsample (27.5k) rather than the full 1.46M; title+content only (`best_answer` unused); no character-level modeling despite noisy spelling; mean-pooled embeddings lose word order; RNN readout is masked mean pooling rather than `pack_padded_sequence`, so the backward direction of Bi- variants still processes padding first.
- **Future work / bonus:** ensemble of the top models; adding `best_answer` text; character-aware or subword models for misspellings; deploy the best model (e.g., Hugging Face Spaces / local Gradio app).

## References

ACL-style starter list — format with `acl.bst` and expand:

- Zhang, X., Zhao, J., and LeCun, Y. 2015. Character-level Convolutional Networks for Text Classification. In *Advances in Neural Information Processing Systems 28 (NIPS 2015)*.
- Devlin, J., Chang, M.-W., Lee, K., and Toutanova, K. 2019. BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. In *Proceedings of NAACL-HLT 2019*.
- Mikolov, T., Chen, K., Corrado, G., and Dean, J. 2013. Efficient Estimation of Word Representations in Vector Space. In *ICLR 2013*. arXiv:1301.3781.
- Pennington, J., Socher, R., and Manning, C. D. 2014. GloVe: Global Vectors for Word Representation. In *Proceedings of EMNLP 2014*.
- Pedregosa, F. et al. 2011. Scikit-learn: Machine Learning in Python. *JMLR 12*.
- Paszke, A. et al. 2019. PyTorch: An Imperative Style, High-Performance Deep Learning Library. In *NeurIPS 2019*.
- Wolf, T. et al. 2020. Transformers: State-of-the-Art Natural Language Processing. In *Proceedings of EMNLP 2020: System Demonstrations*.
- Rehůřek, R. and Sojka, P. 2010. Software Framework for Topic Modelling with Large Corpora. In *LREC 2010*.